In [1]:
import mne
import numpy as np

In [2]:
def get_data(ep_all):
    data = []
    for ii in range(20):
        # 4-class
        ddvf = np.array(ep_all["vf"][ii])
        ddvs = np.array(ep_all["vs"][ii])
        ddam = np.array(ep_all["am"][ii])
        ddas = np.array(ep_all["as"][ii])
        # 2-class
        dda = np.array(ep_all["a"][ii])
        ddv = np.array(ep_all["v"][ii])
        #ddm = np.array(ep_all["m"][ii])[::2]
        X2 = np.concatenate([dda, ddv], axis=0)
        X4 = np.concatenate([ddam, ddas, ddvf, ddvs], axis=0)
        y2 = np.array([0]*len(dda) + [1]*len(ddv))
        y4 = np.array([0]*len(ddam) + [1]*len(ddas) + [2]*len(ddvf) + [3]*len(ddvs))
        data.append([X2, y2, X4, y4])
    return data

In [3]:
epochs = mne.read_epochs("multi/all-epo.fif", preload=True)
epochs.filter(1.,30)
epochs.apply_baseline(baseline=(1.8, 2))  
epochs = epochs.crop(tmin=2, tmax=8)
#epochs.shift_time(-4, relative=True)
chnames = epochs.ch_names
times = epochs.times - 2
subjects = [1,2,5,7,8,9,10,11,12,13,14,16,18,19,21,22,23,24,25,26]
fs = int(epochs.info["sfreq"])
ep_all = {"vf":[],"vs":[],"as":[],"am":[],"v":[],"a":[],"m":[]}
for sub in subjects:
    epvf = epochs[f'category == "face" and subject == {sub}']
    epvs = epochs[f'category == "square" and subject == {sub}']
    epas = epochs[f'category == "speech" and subject == {sub}']
    epam = epochs[f'category == "music" and subject == {sub}']
    epv = epochs[f'modality == "visual" and subject == {sub}']
    epa = epochs[f'modality == "auditory" and subject == {sub}']
    epm = epochs[f'modality == "mix" and subject == {sub}']
    ep_all["v"].append(epv.get_data())
    ep_all["a"].append(epa.get_data())
    ep_all["m"].append(epm.get_data())
    ep_all["vf"].append(epvf.get_data())
    ep_all["vs"].append(epvs.get_data())
    ep_all["as"].append(epas.get_data())
    ep_all["am"].append(epam.get_data())

Reading /home/song-z/Project/multi/all-epo.fif ...
    Found the data of interest:
        t =    -400.00 ...   10000.00 ms
        0 CTF compensation matrices available
Adding metadata with 6 columns
6720 matching events found
No baseline correction applied
0 projection items activated
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 825 samples (3.300 s)

Applying baseline correction (mode: mean)


In [4]:
train_data = get_data(ep_all)

In [10]:
train_data[0][0].shape, train_data[0][1].shape, train_data[0][2].shape, train_data[0][3].shape

((192, 30, 1501), (192,), (192, 30, 1501), (192,))

In [11]:
import pickle
with open("multi/train_data.pkl", "wb") as f:
    pickle.dump(train_data, f)